# Phase 2: Final Improved Baselines - Full Run

## 0. Setup

Same base path style as the original notebook. Update `BASE` if your Drive folder name changed.


In [ ]:


BASE = "/content/drive/MyDrive/Data612 Final Project/Final Data/sovereign_debt_v2_final"

!pip -q install xgboost lightgbm imbalanced-learn tabpfn catboost

import os, json, time, warnings, math, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

from collections import OrderedDict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve
)
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from imblearn.over_sampling import BorderlineSMOTE, ADASYN
import xgboost as xgb

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    HAS_TABPFN = True
except Exception:
    HAS_TABPFN = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}" + (f" - {torch.cuda.get_device_name(0)}" if device.type=='cuda' else ""))

RESULTS_DIR = f"{BASE}/data/final/results_improved_baselines_v2_all_models"
os.makedirs(RESULTS_DIR, exist_ok=True)

XGB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ.setdefault("TABPFN_ALLOW_CPU_LARGE_DATASET", "true")

## 1. Data Loading & Time-Based Split Verification

Train: 2000-2016  
Validation: 2017-2019  
Test: 2020-2024


In [ ]:
print(f"Listing contents of BASE: {BASE}")
!ls -R "{BASE}"

train_df = pd.read_csv(f"{BASE}/data/final/train_flat.csv")
val_df   = pd.read_csv(f"{BASE}/data/final/val_flat.csv")
test_df  = pd.read_csv(f"{BASE}/data/final/test_flat.csv")

for df in [train_df, val_df, test_df]:
    df['year_month'] = pd.to_datetime(df['year_month'])

LABEL = 'distress_within_12m'
ID_COLS = ['iso3', 'year_month', LABEL]
base_feat_cols = [c for c in train_df.columns if c not in ID_COLS]

Xtr_seq = np.load(f"{BASE}/data/final/train_windows.npz")['X'].copy()
ytr_seq = np.load(f"{BASE}/data/final/train_windows.npz")['y'].copy()
Xva_seq = np.load(f"{BASE}/data/final/val_windows.npz")['X'].copy()
yva_seq = np.load(f"{BASE}/data/final/val_windows.npz")['y'].copy()
Xte_seq = np.load(f"{BASE}/data/final/test_windows.npz")['X'].copy()
yte_seq = np.load(f"{BASE}/data/final/test_windows.npz")['y'].copy()

for a in [Xtr_seq, Xva_seq, Xte_seq]:
    a[~np.isfinite(a)] = np.nan

_, SEQ_LEN, N_FEAT_BASE = Xtr_seq.shape

with open(f"{BASE}/data/final/feature_metadata.json") as f:
    meta = json.load(f)

events_df = pd.read_csv(f"{BASE}/data/interim/distress_events.csv")
events_df['start_month'] = pd.to_datetime(events_df['start_month'])

test_meta_seq = pd.read_csv(f"{BASE}/data/final/test_meta.csv")
val_meta_seq  = pd.read_csv(f"{BASE}/data/final/val_meta.csv")
for df in [test_meta_seq, val_meta_seq]:
    if 'window_end' in df.columns:
        df['window_end'] = pd.to_datetime(df['window_end'])

print(f"Base flat features: {len(base_feat_cols)}")
print(f"Base sequence shape: {SEQ_LEN} x {N_FEAT_BASE}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Split verification

Quick leakage check before feature engineering.


In [ ]:
def split_summary(name, df):
    y1 = int(df[LABEL].sum())
    y0 = int((1 - df[LABEL]).sum())
    rate = y1 / max(len(df), 1)
    print(f"{name:<8} {len(df):>6} {y1:>6} {y0:>6} {rate:>7.3%}  {df['year_month'].min().date()} -> {df['year_month'].max().date()}")

print("─"*78)
print(f"{'Split':<8} {'Rows':>6} {'y=1':>6} {'y=0':>6} {'Rate':>8}  Date Range")
print("─"*78)
split_summary("Train", train_df)
split_summary("Val",   val_df)
split_summary("Test",  test_df)
print("─"*78)

print("\nSequence windows")
for name, y, meta_df in [
    ("Train", ytr_seq, None),
    ("Val", yva_seq, val_meta_seq),
    ("Test", yte_seq, test_meta_seq),
]:
    rate = float(np.mean(y))
    if meta_df is not None and 'window_end' in meta_df.columns:
        dr = f"{meta_df['window_end'].min().date()} -> {meta_df['window_end'].max().date()}"
    else:
        dr = "n/a"
    print(f"{name:<8} {len(y):>6} {int(y.sum()):>6} {int((1-y).sum()):>6} {rate:>7.3%}  {dr}")

## 2. Flat Feature Engineering

The original notebook used raw flat features. This version adds backward-looking trend, volatility, rank, and missingness signals.


In [ ]:
all_flat = pd.concat(
    [
        train_df.assign(split='train'),
        val_df.assign(split='val'),
        test_df.assign(split='test')
    ],
    ignore_index=True
).sort_values(['iso3', 'year_month']).reset_index(drop=True)

numeric_base_cols = [c for c in base_feat_cols if pd.api.types.is_numeric_dtype(all_flat[c])]

def add_engineered_features(df, base_cols):
    out = df.copy()
    grouped = out.groupby('iso3', sort=False)

    for col in base_cols:
        s = out[col].astype(float)
        out[f'{col}_missing'] = s.isna().astype(np.int8)

        out[f'{col}_chg1']  = grouped[col].diff(1)
        out[f'{col}_chg3']  = grouped[col].diff(3)
        out[f'{col}_chg6']  = grouped[col].diff(6)
        out[f'{col}_chg12'] = grouped[col].diff(12)

        lag6 = grouped[col].shift(6).replace(0, np.nan)
        out[f'{col}_pctchg6'] = (s - lag6) / (np.abs(lag6) + 1e-6)

        roll6_mean  = grouped[col].transform(lambda x: x.rolling(6,  min_periods=3).mean())
        roll12_mean = grouped[col].transform(lambda x: x.rolling(12, min_periods=6).mean())
        roll12_std  = grouped[col].transform(lambda x: x.rolling(12, min_periods=6).std())

        out[f'{col}_roll6_mean']  = roll6_mean
        out[f'{col}_roll12_mean'] = roll12_mean
        out[f'{col}_roll12_std']  = roll12_std
        out[f'{col}_z12']         = (s - roll12_mean) / (roll12_std + 1e-6)

        ema6 = grouped[col].transform(lambda x: x.ewm(span=6, adjust=False, min_periods=3).mean())
        out[f'{col}_ema6_gap'] = s - ema6

        out[f'{col}_cs_rank'] = out.groupby('year_month')[col].rank(pct=True)
        month_mu = out.groupby('year_month')[col].transform('mean')
        month_sd = out.groupby('year_month')[col].transform('std')
        out[f'{col}_cs_z'] = (s - month_mu) / (month_sd + 1e-6)

    return out

all_flat_fe = add_engineered_features(all_flat, numeric_base_cols)

train_fe = all_flat_fe[all_flat_fe['split'] == 'train'].drop(columns='split').reset_index(drop=True)
val_fe   = all_flat_fe[all_flat_fe['split'] == 'val'].drop(columns='split').reset_index(drop=True)
test_fe  = all_flat_fe[all_flat_fe['split'] == 'test'].drop(columns='split').reset_index(drop=True)

feat_cols = [c for c in train_fe.columns if c not in ID_COLS]
print(f"Numeric base flat features: {len(numeric_base_cols)}")
print(f"Engineered flat features: {len(feat_cols)}")

## 3. Preprocessing & Imbalance Strategy

Class weights remain the main strategy. Borderline-SMOTE is tested only on training flat data.


In [ ]:
X_tr_raw = train_fe[feat_cols].replace([np.inf, -np.inf], np.nan).copy()
X_va_raw = val_fe[feat_cols].replace([np.inf, -np.inf], np.nan).copy()
X_te_raw = test_fe[feat_cols].replace([np.inf, -np.inf], np.nan).copy()

y_tr = train_fe[LABEL].values.astype(int)
y_va = val_fe[LABEL].values.astype(int)
y_te = test_fe[LABEL].values.astype(int)

# Winsorize using train quantiles to reduce extreme outlier impact
q_lo = X_tr_raw.quantile(0.005)
q_hi = X_tr_raw.quantile(0.995)
X_tr_raw = X_tr_raw.clip(lower=q_lo, upper=q_hi, axis=1)
X_va_raw = X_va_raw.clip(lower=q_lo, upper=q_hi, axis=1)
X_te_raw = X_te_raw.clip(lower=q_lo, upper=q_hi, axis=1)

imputer = SimpleImputer(strategy='median')
X_tr_imp = imputer.fit_transform(X_tr_raw)
X_va_imp = imputer.transform(X_va_raw)
X_te_imp = imputer.transform(X_te_raw)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_imp)
X_va_s = scaler.transform(X_va_imp)
X_te_s = scaler.transform(X_te_imp)

# Sequence augmentation: keep raw history, first differences, and missingness masks
seq_nan_tr = np.isnan(Xtr_seq).astype(np.float32)
seq_nan_va = np.isnan(Xva_seq).astype(np.float32)
seq_nan_te = np.isnan(Xte_seq).astype(np.float32)

seq_fill = np.nanmedian(Xtr_seq.reshape(-1, N_FEAT_BASE), axis=0)
for a in [Xtr_seq, Xva_seq, Xte_seq]:
    inds = np.where(np.isnan(a))
    a[inds] = np.take(seq_fill, inds[-1])

def seq_delta(x):
    return np.diff(x, axis=1, prepend=x[:, :1, :])

Xtr_seq_aug = np.concatenate([Xtr_seq, seq_delta(Xtr_seq), seq_nan_tr], axis=2)
Xva_seq_aug = np.concatenate([Xva_seq, seq_delta(Xva_seq), seq_nan_va], axis=2)
Xte_seq_aug = np.concatenate([Xte_seq, seq_delta(Xte_seq), seq_nan_te], axis=2)

seq_mu = Xtr_seq_aug.reshape(-1, Xtr_seq_aug.shape[-1]).mean(axis=0)
seq_sd = Xtr_seq_aug.reshape(-1, Xtr_seq_aug.shape[-1]).std(axis=0) + 1e-8
Xtr_seq_aug = (Xtr_seq_aug - seq_mu) / seq_sd
Xva_seq_aug = (Xva_seq_aug - seq_mu) / seq_sd
Xte_seq_aug = (Xte_seq_aug - seq_mu) / seq_sd

N_FEAT = Xtr_seq_aug.shape[-1]

POS_WEIGHT_FLAT = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
POS_WEIGHT_SEQ  = (ytr_seq == 0).sum() / max((ytr_seq == 1).sum(), 1)

smote = BorderlineSMOTE(random_state=SEED, k_neighbors=3)
adasyn = ADASYN(random_state=SEED, n_neighbors=3)

X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_s, y_tr)
X_tr_ad, y_tr_ad = adasyn.fit_resample(X_tr_s, y_tr)

# TabPFN works better with a smaller feature set
TABPFN_K = min(256, X_tr_imp.shape[1])
tab_selector = SelectKBest(score_func=mutual_info_classif, k=TABPFN_K)
X_tr_tab = tab_selector.fit_transform(X_tr_imp, y_tr)
X_va_tab = tab_selector.transform(X_va_imp)
X_te_tab = tab_selector.transform(X_te_imp)

print(f"Flat pos weight: {POS_WEIGHT_FLAT:.2f}")
print(f"Seq pos weight : {POS_WEIGHT_SEQ:.2f}")
print(f"Augmented sequence shape: {SEQ_LEN} x {N_FEAT}")
print(f"TabPFN feature count: {X_tr_tab.shape[1]}")

## 4. Evaluation Framework

Metrics, calibration, confidence intervals, lead-time, and one shared evaluation function.


In [ ]:
def find_best_threshold(y_true, y_prob):
    pr, rc, ths = precision_recall_curve(y_true, y_prob)
    if len(ths) == 0:
        return 0.5
    f1s = 2 * pr[:-1] * rc[:-1] / (pr[:-1] + rc[:-1] + 1e-8)
    top = np.argsort(f1s)[-10:]
    best_th, best_score = 0.5, -1
    k = max(int(np.asarray(y_true).sum()), 1)
    for idx in top:
        th = float(ths[idx])
        pred = (np.asarray(y_prob) >= th).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        p = precision_score(y_true, pred, zero_division=0)
        top_k_idx = np.argsort(y_prob)[-k:]
        pk = np.asarray(y_true)[top_k_idx].mean()
        score = f1 + 0.15 * p + 0.10 * pk
        if score > best_score:
            best_score = score
            best_th = th
    return best_th

def compute_all_metrics(y_true, y_prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if threshold is None:
        threshold = find_best_threshold(y_true, y_prob)

    y_pred = (y_prob >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    auprc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    f1  = f1_score(y_true, y_pred, zero_division=0)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    k = max(int(y_true.sum()), 1)
    top_k_idx = np.argsort(y_prob)[-k:]
    pk = y_true[top_k_idx].mean()
    rk = y_true[top_k_idx].sum() / max(y_true.sum(), 1)

    return {
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1": f1,
        "Precision": pre,
        "Recall": rec,
        "P@k": pk,
        "R@k": rk,
        "Threshold": threshold
    }

def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=500, ci=0.95, seed=42):
    rng = np.random.RandomState(seed)
    scores = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        yt = y_true[idx]
        yp = y_prob[idx]
        try:
            if len(np.unique(yt)) > 1:
                scores.append(metric_fn(yt, yp))
        except Exception:
            pass
    if len(scores) == 0:
        return (np.nan, np.nan)
    lo = np.percentile(scores, (1 - ci) / 2 * 100)
    hi = np.percentile(scores, (1 + ci) / 2 * 100)
    return (float(lo), float(hi))

def bootstrap_all_metrics(y_true, y_prob):
    return {
        "AUROC_CI": bootstrap_ci(y_true, y_prob, roc_auc_score),
        "AUPRC_CI": bootstrap_ci(y_true, y_prob, average_precision_score)
    }

def choose_calibration(y_val, val_prob, test_prob, val_dates=None):
    y_val = np.asarray(y_val).astype(int)
    val_prob = np.asarray(val_prob).astype(float)
    test_prob = np.asarray(test_prob).astype(float)

    if val_dates is None:
        order = np.arange(len(y_val))
    else:
        order = np.argsort(pd.to_datetime(val_dates).values)

    if len(order) < 50 or y_val.sum() < 12:
        return val_prob, test_prob, "raw"

    split = max(int(0.60 * len(order)), 30)
    cal_idx = order[:split]
    sel_idx = order[split:]

    if y_val[cal_idx].sum() < 6 or y_val[sel_idx].sum() < 6:
        return val_prob, test_prob, "raw"

    candidates = {}

    candidates["raw"] = (
        val_prob.copy(),
        test_prob.copy()
    )

    try:
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
        iso.fit(val_prob[cal_idx], y_val[cal_idx])
        candidates["isotonic"] = (
            iso.transform(val_prob),
            iso.transform(test_prob)
        )
    except Exception:
        pass

    try:
        cal_lr = LogisticRegression(solver='lbfgs', max_iter=2000)
        cal_lr.fit(val_prob[cal_idx].reshape(-1, 1), y_val[cal_idx])
        candidates["platt"] = (
            cal_lr.predict_proba(val_prob.reshape(-1, 1))[:, 1],
            cal_lr.predict_proba(test_prob.reshape(-1, 1))[:, 1]
        )
    except Exception:
        pass

    best_name, best_pair, best_ap, best_auc = "raw", candidates["raw"], -1, -1
    for name, (vprob, tprob) in candidates.items():
        ap = average_precision_score(y_val[sel_idx], vprob[sel_idx])
        auc = roc_auc_score(y_val[sel_idx], vprob[sel_idx])
        if (ap > best_ap) or (ap == best_ap and auc > best_auc):
            best_name, best_pair, best_ap, best_auc = name, (vprob, tprob), ap, auc

    return best_pair[0], best_pair[1], best_name

def compute_lead_times(pred_df, events, threshold, date_col='year_month'):
    pred_df = pred_df.copy()
    pred_df[date_col] = pd.to_datetime(pred_df[date_col])
    pred_df['flagged'] = (pred_df['prob'] >= threshold).astype(int)

    ev = events.copy()
    ev['start_month'] = pd.to_datetime(ev['start_month'])

    pred_min, pred_max = pred_df[date_col].min(), pred_df[date_col].max()
    ev = ev[(ev['start_month'] >= pred_min) & (ev['start_month'] <= pred_max)]

    lead_times, event_rows = [], []
    for _, row in ev.iterrows():
        iso3, start = row['iso3'], row['start_month']
        lookback = start - pd.DateOffset(months=12)
        mask = (
            (pred_df['iso3'] == iso3) &
            (pred_df[date_col] >= lookback) &
            (pred_df[date_col] < start) &
            (pred_df['flagged'] == 1)
        )
        flagged = pred_df.loc[mask]
        if len(flagged) > 0:
            first_flag = flagged[date_col].min()
            lead = (start - first_flag).days / 30.44
            lead_times.append(lead)
            event_rows.append({
                "iso3": iso3,
                "crisis_date": str(start.date()),
                "first_flag": str(first_flag.date()),
                "lead_months": round(lead, 1),
                "detected": True
            })
        else:
            event_rows.append({
                "iso3": iso3,
                "crisis_date": str(start.date()),
                "first_flag": None,
                "lead_months": 0.0,
                "detected": False
            })
    return lead_times, pd.DataFrame(event_rows)

def full_evaluate(name, y_val, val_prob, y_test, test_prob, pred_df=None, events=None, calibration_name="raw"):
    val_metrics = compute_all_metrics(y_val, val_prob)
    threshold = val_metrics['Threshold']
    test_metrics = compute_all_metrics(y_test, test_prob, threshold=threshold)
    cis = bootstrap_all_metrics(y_test, test_prob)

    lead_times, lead_df = [], pd.DataFrame()
    if pred_df is not None and events is not None:
        lead_times, lead_df = compute_lead_times(pred_df, events, threshold)

    print(f"\n{'═'*70}")
    print(name)
    print(f"{'═'*70}")
    print(f"Calibration -> {calibration_name}")
    print(f"Validation -> AUROC: {val_metrics['AUROC']:.4f} | AUPRC: {val_metrics['AUPRC']:.4f} | F1: {val_metrics['F1']:.4f} | Threshold: {threshold:.4f}")
    print(f"Test       -> AUROC: {test_metrics['AUROC']:.4f} | AUPRC: {test_metrics['AUPRC']:.4f} | F1: {test_metrics['F1']:.4f} | P@k: {test_metrics['P@k']:.4f}")

    return {
        "name": name,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "cis": cis,
        "lead_times": lead_times,
        "lead_df": lead_df,
        "calibration": calibration_name
    }

def make_pred_df(base_df, label_col, prob):
    out = base_df[['iso3', 'year_month', label_col]].copy()
    out = out.rename(columns={label_col: 'y_true'})
    out['prob'] = np.asarray(prob)
    return out

pred_store = OrderedDict()
all_evals = []

## 5. Model 1 - Logistic Regression

Elastic-net search, AUPRC-based selection, Borderline-SMOTE check, isotonic calibration.


In [ ]:
t0 = time.time()

lr_configs = [
    {"penalty": "l2", "C": 0.01, "cw_mult": 0.75, "variant": "weighted"},
    {"penalty": "l2", "C": 0.05, "cw_mult": 1.00, "variant": "weighted"},
    {"penalty": "l2", "C": 0.10, "cw_mult": 1.25, "variant": "weighted"},
    {"penalty": "l2", "C": 0.50, "cw_mult": 1.50, "variant": "weighted"},
    {"penalty": "l1", "C": 0.05, "cw_mult": 1.00, "variant": "weighted"},
    {"penalty": "l1", "C": 0.10, "cw_mult": 1.25, "variant": "weighted"},
    {"penalty": "elasticnet", "C": 0.05, "l1_ratio": 0.30, "cw_mult": 1.00, "variant": "weighted"},
    {"penalty": "elasticnet", "C": 0.10, "l1_ratio": 0.50, "cw_mult": 1.25, "variant": "weighted"},
    {"penalty": "elasticnet", "C": 0.20, "l1_ratio": 0.70, "cw_mult": 1.50, "variant": "weighted"},
    {"penalty": "l2", "C": 0.05, "variant": "smote"},
    {"penalty": "l2", "C": 0.10, "variant": "smote"},
    {"penalty": "elasticnet", "C": 0.10, "l1_ratio": 0.50, "variant": "adasyn"},
]

best_lr_ap, best_lr_auc, best_lr, best_lr_cfg = -1, -1, None, None
print("Logistic Regression search")
for cfg in lr_configs:
    variant = cfg["variant"]
    if variant == "weighted":
        X_fit, y_fit = X_tr_s, y_tr
        cw = {0: 1.0, 1: float(POS_WEIGHT_FLAT * cfg.get("cw_mult", 1.0))}
    elif variant == "smote":
        X_fit, y_fit = X_tr_sm, y_tr_sm
        cw = None
    else:
        X_fit, y_fit = X_tr_ad, y_tr_ad
        cw = None

    model = LogisticRegression(
        penalty=cfg["penalty"],
        C=cfg["C"],
        l1_ratio=cfg.get("l1_ratio", None),
        solver='saga',
        class_weight=cw,
        max_iter=7000,
        random_state=SEED,
        n_jobs=-1
    )
    model.fit(X_fit, y_fit)
    prob_v = model.predict_proba(X_va_s)[:, 1]
    ap = average_precision_score(y_va, prob_v)
    auc = roc_auc_score(y_va, prob_v)
    print(f"  {cfg} -> AUROC: {auc:.4f}  AUPRC: {ap:.4f}")
    if (ap > best_lr_ap) or (ap == best_lr_ap and auc > best_lr_auc):
        best_lr_ap, best_lr_auc, best_lr, best_lr_cfg = ap, auc, model, cfg

print(f"\nBest LR config: {best_lr_cfg}")

lr_val_raw  = best_lr.predict_proba(X_va_s)[:, 1]
lr_test_raw = best_lr.predict_proba(X_te_s)[:, 1]
lr_val_prob, lr_test_prob, lr_cal = choose_calibration(y_va, lr_val_raw, lr_test_raw, val_fe['year_month'])

lr_val_df  = make_pred_df(val_fe, LABEL, lr_val_prob)
lr_test_df = make_pred_df(test_fe, LABEL, lr_test_prob)

lr_eval = full_evaluate(
    "Logistic Regression",
    y_va, lr_val_prob, y_te, lr_test_prob,
    lr_test_df[['iso3', 'year_month', 'prob']], events_df,
    calibration_name=lr_cal
)
lr_eval['time'] = time.time() - t0
all_evals.append(lr_eval)
pred_store["Logistic Regression"] = {"val": lr_val_df, "test": lr_test_df}

## 6. Model 2 - XGBoost

In [ ]:
t0 = time.time()

xgb_configs = [
    {"booster": "gbtree", "max_depth": 2, "n_estimators": 500, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 4, "gamma": 0.0, "reg_alpha": 0.0, "reg_lambda": 1.0, "max_delta_step": 1},
    {"booster": "gbtree", "max_depth": 3, "n_estimators": 700, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 8, "gamma": 0.5, "reg_alpha": 0.3, "reg_lambda": 1.5, "max_delta_step": 2},
    {"booster": "gbtree", "max_depth": 4, "n_estimators": 600, "learning_rate": 0.03, "subsample": 0.75, "colsample_bytree": 0.75, "min_child_weight": 10, "gamma": 1.0, "reg_alpha": 0.7, "reg_lambda": 2.0, "max_delta_step": 2},
    {"booster": "gbtree", "max_depth": 5, "n_estimators": 450, "learning_rate": 0.03, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_weight": 12, "gamma": 2.0, "reg_alpha": 1.2, "reg_lambda": 3.0, "max_delta_step": 3},
    {"booster": "dart",   "max_depth": 3, "n_estimators": 500, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_weight": 6, "gamma": 1.0, "reg_alpha": 0.5, "reg_lambda": 2.0, "rate_drop": 0.10, "skip_drop": 0.40, "max_delta_step": 2},
    {"booster": "dart",   "max_depth": 4, "n_estimators": 600, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.75, "min_child_weight": 8, "gamma": 1.5, "reg_alpha": 0.8, "reg_lambda": 2.5, "rate_drop": 0.15, "skip_drop": 0.30, "max_delta_step": 2},
]

best_xgb_ap, best_xgb_auc, best_xgb, best_xgb_cfg = -1, -1, None, None
print("XGBoost search")
for cfg in xgb_configs:
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        scale_pos_weight=POS_WEIGHT_FLAT,
        early_stopping_rounds=60,
        random_state=SEED,
        verbosity=0,
        tree_method='hist',
        device=XGB_DEVICE,
        **cfg
    )
    model.fit(X_tr_imp, y_tr, eval_set=[(X_va_imp, y_va)], verbose=False)
    prob_v = model.predict_proba(X_va_imp)[:, 1]
    ap = average_precision_score(y_va, prob_v)
    auc = roc_auc_score(y_va, prob_v)
    print(f"  {cfg} -> AUROC: {auc:.4f}  AUPRC: {ap:.4f}")
    if (ap > best_xgb_ap) or (ap == best_xgb_ap and auc > best_xgb_auc):
        best_xgb_ap, best_xgb_auc, best_xgb, best_xgb_cfg = ap, auc, model, cfg

print(f"\nBest XGBoost config: {best_xgb_cfg}")

xgb_val_raw  = best_xgb.predict_proba(X_va_imp)[:, 1]
xgb_test_raw = best_xgb.predict_proba(X_te_imp)[:, 1]
xgb_val_prob, xgb_test_prob, xgb_cal = choose_calibration(y_va, xgb_val_raw, xgb_test_raw, val_fe['year_month'])

xgb_val_df  = make_pred_df(val_fe, LABEL, xgb_val_prob)
xgb_test_df = make_pred_df(test_fe, LABEL, xgb_test_prob)

xgb_eval = full_evaluate(
    "XGBoost",
    y_va, xgb_val_prob, y_te, xgb_test_prob,
    xgb_test_df[['iso3', 'year_month', 'prob']], events_df,
    calibration_name=xgb_cal
)
xgb_eval['time'] = time.time() - t0
all_evals.append(xgb_eval)
pred_store["XGBoost"] = {"val": xgb_val_df, "test": xgb_test_df}

## 7. Model 3 - LightGBM

In [ ]:
t0 = time.time()

if HAS_LGBM:
    lgbm_configs = [
        {"boosting_type": "gbdt", "num_leaves": 15, "max_depth": 4, "n_estimators": 500, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_samples": 20, "reg_alpha": 0.0, "reg_lambda": 1.0},
        {"boosting_type": "gbdt", "num_leaves": 31, "max_depth": 5, "n_estimators": 700, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.75, "min_child_samples": 30, "reg_alpha": 0.3, "reg_lambda": 1.5},
        {"boosting_type": "gbdt", "num_leaves": 63, "max_depth": 6, "n_estimators": 800, "learning_rate": 0.02, "subsample": 0.75, "colsample_bytree": 0.70, "min_child_samples": 40, "reg_alpha": 0.7, "reg_lambda": 2.0},
        {"boosting_type": "dart", "num_leaves": 31, "max_depth": 5, "n_estimators": 600, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_samples": 20, "reg_alpha": 0.2, "reg_lambda": 1.5},
        {"boosting_type": "dart", "num_leaves": 63, "max_depth": 6, "n_estimators": 700, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.75, "min_child_samples": 30, "reg_alpha": 0.5, "reg_lambda": 2.0},
    ]

    best_lgb_ap, best_lgb_auc, best_lgbm, best_lgb_cfg = -1, -1, None, None
    print("LightGBM search")
    for cfg in lgbm_configs:
        model = lgb.LGBMClassifier(
            objective='binary',
            scale_pos_weight=POS_WEIGHT_FLAT,
            random_state=SEED,
            verbosity=-1,
            **cfg
        )
        model.fit(
            X_tr_imp, y_tr,
            eval_set=[(X_va_imp, y_va)],
            eval_metric='average_precision',
            callbacks=[lgb.early_stopping(60, verbose=False)]
        )
        prob_v = model.predict_proba(X_va_imp)[:, 1]
        ap = average_precision_score(y_va, prob_v)
        auc = roc_auc_score(y_va, prob_v)
        print(f"  {cfg} -> AUROC: {auc:.4f}  AUPRC: {ap:.4f}")
        if (ap > best_lgb_ap) or (ap == best_lgb_ap and auc > best_lgb_auc):
            best_lgb_ap, best_lgb_auc, best_lgbm, best_lgb_cfg = ap, auc, model, cfg

    print(f"\nBest LightGBM config: {best_lgb_cfg}")

    lgb_val_raw  = best_lgbm.predict_proba(X_va_imp)[:, 1]
    lgb_test_raw = best_lgbm.predict_proba(X_te_imp)[:, 1]
    lgb_val_prob, lgb_test_prob, lgb_cal = choose_calibration(y_va, lgb_val_raw, lgb_test_raw, val_fe['year_month'])

    lgb_val_df  = make_pred_df(val_fe, LABEL, lgb_val_prob)
    lgb_test_df = make_pred_df(test_fe, LABEL, lgb_test_prob)

    lgb_eval = full_evaluate(
        "LightGBM",
        y_va, lgb_val_prob, y_te, lgb_test_prob,
        lgb_test_df[['iso3', 'year_month', 'prob']], events_df,
        calibration_name=lgb_cal
    )
    lgb_eval['time'] = time.time() - t0
    all_evals.append(lgb_eval)
    pred_store["LightGBM"] = {"val": lgb_val_df, "test": lgb_test_df}
else:
    print("LightGBM not available in this runtime.")

## 8. Shared Neural Training Infrastructure

Reusable loop for the sequence baselines.


## 8. Model 4 - CatBoost (optional strong tabular baseline)

In [ ]:
t0 = time.time()

if HAS_CAT:
    cat_configs = [
        {"depth": 4, "iterations": 500, "learning_rate": 0.03, "l2_leaf_reg": 3.0},
        {"depth": 5, "iterations": 700, "learning_rate": 0.02, "l2_leaf_reg": 5.0},
        {"depth": 6, "iterations": 600, "learning_rate": 0.03, "l2_leaf_reg": 7.0},
    ]

    best_cat_ap, best_cat_auc, best_cat, best_cat_cfg = -1, -1, None, None
    print("CatBoost search")
    for cfg in cat_configs:
        model = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='PRAUC',
            random_seed=SEED,
            verbose=False,
            class_weights=[1.0, float(POS_WEIGHT_FLAT)],
            **cfg
        )
        model.fit(X_tr_imp, y_tr, eval_set=(X_va_imp, y_va), use_best_model=True, verbose=False)
        prob_v = model.predict_proba(X_va_imp)[:, 1]
        ap = average_precision_score(y_va, prob_v)
        auc = roc_auc_score(y_va, prob_v)
        print(f"  {cfg} -> AUROC: {auc:.4f}  AUPRC: {ap:.4f}")
        if (ap > best_cat_ap) or (ap == best_cat_ap and auc > best_cat_auc):
            best_cat_ap, best_cat_auc, best_cat, best_cat_cfg = ap, auc, model, cfg

    print(f"\nBest CatBoost config: {best_cat_cfg}")

    cat_val_raw  = best_cat.predict_proba(X_va_imp)[:, 1]
    cat_test_raw = best_cat.predict_proba(X_te_imp)[:, 1]
    cat_val_prob, cat_test_prob, cat_cal = choose_calibration(y_va, cat_val_raw, cat_test_raw, val_fe['year_month'])

    cat_val_df  = make_pred_df(val_fe, LABEL, cat_val_prob)
    cat_test_df = make_pred_df(test_fe, LABEL, cat_test_prob)

    cat_eval = full_evaluate(
        "CatBoost",
        y_va, cat_val_prob, y_te, cat_test_prob,
        cat_test_df[['iso3', 'year_month', 'prob']], events_df,
        calibration_name=cat_cal
    )
    cat_eval['time'] = time.time() - t0
    all_evals.append(cat_eval)
    pred_store["CatBoost"] = {"val": cat_val_df, "test": cat_test_df}
else:
    print("CatBoost not available in this runtime.")

In [ ]:
class FocalBCEWithLogits(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none', pos_weight=self.pos_weight
        )
        pt = torch.exp(-bce)
        return ((1 - pt) ** self.gamma * bce).mean()

def train_nn(model, Xtr, ytr, Xva, yva, epochs=90, lr=1e-3, patience=14, batch=96, loss_name='focal', name='NN'):
    pos_w = torch.tensor([POS_WEIGHT_SEQ], dtype=torch.float32, device=device)
    crit = FocalBCEWithLogits(pos_weight=pos_w, gamma=2.0) if loss_name == 'focal' else nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 10))

    sample_w = np.where(ytr == 1, POS_WEIGHT_SEQ, 1.0).astype(np.float32)
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_w), num_samples=len(sample_w), replacement=True)

    tl = DataLoader(
        TensorDataset(torch.FloatTensor(Xtr), torch.FloatTensor(ytr.astype(np.float32))),
        batch_size=batch, sampler=sampler, shuffle=False
    )
    vl = DataLoader(
        TensorDataset(torch.FloatTensor(Xva), torch.FloatTensor(yva.astype(np.float32))),
        batch_size=256, shuffle=False
    )

    best_ap, best_state, wait = -1, None, 0
    hist = []
    for ep in range(epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item()

        sched.step()

        model.eval()
        val_prob = []
        with torch.no_grad():
            for xb, _ in vl:
                xb = xb.to(device)
                val_prob.append(torch.sigmoid(model(xb)).cpu().numpy())
        val_prob = np.concatenate(val_prob)

        val_ap = average_precision_score(yva, val_prob)
        val_auc = roc_auc_score(yva, val_prob)
        hist.append({"epoch": ep + 1, "loss": train_loss / max(len(tl), 1), "val_auroc": val_auc, "val_auprc": val_ap})

        if val_ap > best_ap:
            best_ap = val_ap
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (ep + 1) % 10 == 0:
            print(f"  [{name}] epoch {ep+1:>3} | loss {train_loss / max(len(tl), 1):.4f} | AUROC {val_auc:.4f} | AUPRC {val_ap:.4f}")

        if wait >= patience:
            print(f"  [{name}] early stop at epoch {ep+1}")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

def predict_nn(model, X):
    model.eval()
    dl = DataLoader(TensorDataset(torch.FloatTensor(X), torch.zeros(len(X))), batch_size=256, shuffle=False)
    probs = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(probs)

def make_seq_pred_df(meta_df, y_true, prob):
    out = meta_df[['iso3']].copy()
    out['year_month'] = pd.to_datetime(meta_df['window_end'])
    out['y_true'] = np.asarray(y_true).astype(int)
    out['prob'] = np.asarray(prob).astype(float)
    return out

## 9. Model 4 - LSTM

Original baseline family, but with stronger tuning and focal loss.


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, inp, hid=128, layers=2, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(inp, hid, layers, batch_first=True, dropout=drop if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hid * 2)
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(hid * 2, 64),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        mean_pool = out.mean(dim=1)
        max_pool, _ = out.max(dim=1)
        pooled = torch.cat([mean_pool, max_pool], dim=1)
        pooled = self.norm(pooled)
        return self.head(pooled).squeeze(-1)

t0 = time.time()

lstm_configs = [
    {"hid": 64,  "drop": 0.20, "lr": 8e-4, "loss_name": "focal"},
    {"hid": 96,  "drop": 0.25, "lr": 6e-4, "loss_name": "focal"},
    {"hid": 128, "drop": 0.30, "lr": 5e-4, "loss_name": "focal"},
    {"hid": 128, "drop": 0.35, "lr": 4e-4, "loss_name": "bce"},
]

best_lstm_ap, best_lstm, best_lstm_hist, best_lstm_cfg = -1, None, None, None
print("LSTM search")
for cfg in lstm_configs:
    print(f"  config -> {cfg}")
    model = LSTMModel(N_FEAT, hid=cfg['hid'], drop=cfg['drop']).to(device)
    model, hist = train_nn(
        model,
        Xtr_seq_aug, ytr_seq, Xva_seq_aug, yva_seq,
        lr=cfg['lr'], patience=14, loss_name=cfg['loss_name'], name='LSTM'
    )
    prob_v = predict_nn(model, Xva_seq_aug)
    ap = average_precision_score(yva_seq, prob_v)
    if ap > best_lstm_ap:
        best_lstm_ap, best_lstm, best_lstm_hist, best_lstm_cfg = ap, model, hist, cfg

print(f"\nBest LSTM config: {best_lstm_cfg}")

lstm_val_raw  = predict_nn(best_lstm, Xva_seq_aug)
lstm_test_raw = predict_nn(best_lstm, Xte_seq_aug)
lstm_val_prob, lstm_test_prob, lstm_cal = choose_calibration(yva_seq, lstm_val_raw, lstm_test_raw, val_meta_seq['window_end'])

lstm_val_df  = make_seq_pred_df(val_meta_seq, yva_seq, lstm_val_prob)
lstm_test_df = make_seq_pred_df(test_meta_seq, yte_seq, lstm_test_prob)

lstm_eval = full_evaluate(
    "LSTM",
    yva_seq, lstm_val_prob, yte_seq, lstm_test_prob,
    lstm_test_df[['iso3','year_month','prob']], events_df,
    calibration_name=lstm_cal
)
lstm_eval['time'] = time.time() - t0
all_evals.append(lstm_eval)
pred_store["LSTM"] = {"val": lstm_val_df, "test": lstm_test_df}

## 10. Model 5 - BiLSTM + Attention

Stronger temporal pooling baseline for the same sequence data.


In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x).squeeze(-1), dim=1)
        return torch.sum(x * w.unsqueeze(-1), dim=1)

class BiLSTMAttn(nn.Module):
    def __init__(self, inp, hid=96, layers=2, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            inp, hid, layers,
            batch_first=True,
            dropout=drop if layers > 1 else 0.0,
            bidirectional=True
        )
        self.pool = AttentionPool(hid * 2)
        self.norm = nn.LayerNorm(hid * 2)
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(hid * 2, 64),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        pooled = self.pool(out)
        pooled = self.norm(pooled)
        return self.head(pooled).squeeze(-1)

t0 = time.time()

bilstm_configs = [
    {"hid": 64,  "drop": 0.20, "lr": 7e-4, "loss_name": "focal"},
    {"hid": 96,  "drop": 0.25, "lr": 5e-4, "loss_name": "focal"},
    {"hid": 128, "drop": 0.30, "lr": 4e-4, "loss_name": "focal"},
    {"hid": 128, "drop": 0.35, "lr": 3e-4, "loss_name": "bce"},
]

best_bilstm_ap, best_bilstm, best_bilstm_cfg = -1, None, None
print("BiLSTM + Attention search")
for cfg in bilstm_configs:
    print(f"  config -> {cfg}")
    model = BiLSTMAttn(N_FEAT, hid=cfg['hid'], drop=cfg['drop']).to(device)
    model, hist = train_nn(
        model,
        Xtr_seq_aug, ytr_seq, Xva_seq_aug, yva_seq,
        lr=cfg['lr'], patience=14, loss_name=cfg['loss_name'], name='BiLSTM'
    )
    prob_v = predict_nn(model, Xva_seq_aug)
    ap = average_precision_score(yva_seq, prob_v)
    if ap > best_bilstm_ap:
        best_bilstm_ap, best_bilstm, best_bilstm_cfg = ap, model, cfg

print(f"\nBest BiLSTM config: {best_bilstm_cfg}")

bilstm_val_raw  = predict_nn(best_bilstm, Xva_seq_aug)
bilstm_test_raw = predict_nn(best_bilstm, Xte_seq_aug)
bilstm_val_prob, bilstm_test_prob, bilstm_cal = choose_calibration(yva_seq, bilstm_val_raw, bilstm_test_raw, val_meta_seq['window_end'])

bilstm_val_df  = make_seq_pred_df(val_meta_seq, yva_seq, bilstm_val_prob)
bilstm_test_df = make_seq_pred_df(test_meta_seq, yte_seq, bilstm_test_prob)

bilstm_eval = full_evaluate(
    "BiLSTM + Attention",
    yva_seq, bilstm_val_prob, yte_seq, bilstm_test_prob,
    bilstm_test_df[['iso3','year_month','prob']], events_df,
    calibration_name=bilstm_cal
)
bilstm_eval['time'] = time.time() - t0
all_evals.append(bilstm_eval)
pred_store["BiLSTM + Attention"] = {"val": bilstm_val_df, "test": bilstm_test_df}

## 11. Model 6 - Transformer Encoder

Transformer baseline kept from the original notebook with the stronger training stack.


In [ ]:
class TransformerAttnPool(nn.Module):
    def __init__(self, inp, d_model=96, nhead=4, nlayers=2, drop=0.3, seq_len=24):
        super().__init__()
        self.input_norm = nn.LayerNorm(inp)
        self.proj = nn.Linear(inp, d_model)
        self.pos = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=drop,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.pool = AttentionPool(d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_norm(x)
        x = self.proj(x) + self.pos[:, :x.size(1), :]
        x = self.enc(x)
        x = self.pool(x)
        x = self.norm(x)
        return self.head(x).squeeze(-1)

t0 = time.time()

tf_configs = [
    {"d_model": 64,  "nhead": 4, "nlayers": 2, "drop": 0.20, "lr": 6e-4, "loss_name": "focal"},
    {"d_model": 96,  "nhead": 4, "nlayers": 2, "drop": 0.25, "lr": 4e-4, "loss_name": "focal"},
    {"d_model": 128, "nhead": 4, "nlayers": 3, "drop": 0.30, "lr": 3e-4, "loss_name": "focal"},
    {"d_model": 96,  "nhead": 8, "nlayers": 2, "drop": 0.30, "lr": 4e-4, "loss_name": "bce"},
]

best_tf_ap, best_tf, best_tf_cfg = -1, None, None
print("Transformer search")
for cfg in tf_configs:
    print(f"  config -> {cfg}")
    model = TransformerAttnPool(
        N_FEAT,
        d_model=cfg['d_model'],
        nhead=cfg['nhead'],
        nlayers=cfg['nlayers'],
        drop=cfg['drop'],
        seq_len=SEQ_LEN
    ).to(device)
    model, hist = train_nn(
        model,
        Xtr_seq_aug, ytr_seq, Xva_seq_aug, yva_seq,
        lr=cfg['lr'], patience=12, loss_name=cfg['loss_name'], name='Transformer'
    )
    prob_v = predict_nn(model, Xva_seq_aug)
    ap = average_precision_score(yva_seq, prob_v)
    if ap > best_tf_ap:
        best_tf_ap, best_tf, best_tf_cfg = ap, model, cfg

print(f"\nBest Transformer config: {best_tf_cfg}")

tf_val_raw  = predict_nn(best_tf, Xva_seq_aug)
tf_test_raw = predict_nn(best_tf, Xte_seq_aug)
tf_val_prob, tf_test_prob, tf_cal = choose_calibration(yva_seq, tf_val_raw, tf_test_raw, val_meta_seq['window_end'])

tf_val_df  = make_seq_pred_df(val_meta_seq, yva_seq, tf_val_prob)
tf_test_df = make_seq_pred_df(test_meta_seq, yte_seq, tf_test_prob)

tf_eval = full_evaluate(
    "Transformer Encoder",
    yva_seq, tf_val_prob, yte_seq, tf_test_prob,
    tf_test_df[['iso3','year_month','prob']], events_df,
    calibration_name=tf_cal
)
tf_eval['time'] = time.time() - t0
all_evals.append(tf_eval)
pred_store["Transformer Encoder"] = {"val": tf_val_df, "test": tf_test_df}

## 12. Model 7 - TabPFN-2.5

Post-2024 tabular foundation baseline. It uses the engineered flat features without scaling.


In [ ]:
t0 = time.time()

if HAS_TABPFN:
    try:
        tab_model = TabPFNClassifier.create_default_for_version(ModelVersion.V2_5)
        tab_model.fit(X_tr_tab, y_tr)

        tab_val_raw  = tab_model.predict_proba(X_va_tab)[:, 1]
        tab_test_raw = tab_model.predict_proba(X_te_tab)[:, 1]
        tab_val_prob, tab_test_prob, tab_cal = choose_calibration(y_va, tab_val_raw, tab_test_raw, val_fe['year_month'])

        tab_val_df  = make_pred_df(val_fe, LABEL, tab_val_prob)
        tab_test_df = make_pred_df(test_fe, LABEL, tab_test_prob)

        tab_eval = full_evaluate(
            "TabPFN-2.5",
            y_va, tab_val_prob, y_te, tab_test_prob,
            tab_test_df[['iso3','year_month','prob']], events_df,
            calibration_name=tab_cal
        )
        tab_eval['time'] = time.time() - t0
        all_evals.append(tab_eval)
        pred_store["TabPFN-2.5"] = {"val": tab_val_df, "test": tab_test_df}
    except Exception as e:
        print(f"TabPFN-2.5 skipped: {e}")
else:
    print("TabPFN not available in this runtime.")

## 13. Rank-Average Ensemble

Merge available flat and sequence models on common rows and average percentile ranks.


In [ ]:
def build_weighted_rank_ensemble(pred_store, evals, split='test', keep_top=5):
    metric_map = {ev['name']: ev['val_metrics']['AUPRC'] for ev in evals}
    baseline_ap = float(np.mean([bundle['val']['y_true'].mean() for bundle in pred_store.values()]))

    items = []
    for name, bundle in pred_store.items():
        score = metric_map.get(name, baseline_ap)
        weight = max(score - baseline_ap, 1e-4)
        items.append((name, weight, bundle))

    items = sorted(items, key=lambda x: x[1], reverse=True)[:min(keep_top, len(items))]

    frames, weights = [], []
    for name, weight, bundle in items:
        df = bundle[split][['iso3', 'year_month', 'y_true', 'prob']].copy()
        df = df.rename(columns={'prob': f'prob_{name}'})
        frames.append(df)
        weights.append(weight)

    merged = frames[0]
    for df in frames[1:]:
        merged = merged.merge(df, on=['iso3', 'year_month', 'y_true'], how='inner')

    prob_cols = [c for c in merged.columns if c.startswith('prob_')]
    rank_df = merged[prob_cols].rank(pct=True)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    merged['prob'] = np.average(rank_df.values, axis=1, weights=weights)
    return merged[['iso3', 'year_month', 'y_true', 'prob']], prob_cols, weights

ens_val_df, ens_val_cols, ens_w = build_weighted_rank_ensemble(pred_store, all_evals, split='val', keep_top=5)
ens_test_df, ens_test_cols, _   = build_weighted_rank_ensemble(pred_store, all_evals, split='test', keep_top=5)

print(f"Ensemble models: {ens_val_cols}")
print(f"Validation rows used: {len(ens_val_df)}")
print(f"Test rows used: {len(ens_test_df)}")
print(f"Ensemble weights: {dict(zip(ens_val_cols, np.round(ens_w, 4)))}")

ens_eval = full_evaluate(
    "Weighted Rank Ensemble",
    ens_val_df['y_true'].values,
    ens_val_df['prob'].values,
    ens_test_df['y_true'].values,
    ens_test_df['prob'].values,
    ens_test_df[['iso3', 'year_month', 'prob']],
    events_df,
    calibration_name="rank-average"
)
ens_eval['time'] = np.nan
all_evals.append(ens_eval)
pred_store["Weighted Rank Ensemble"] = {"val": ens_val_df, "test": ens_test_df}

## 14. Model Comparison Table

Validation and test summaries for every model that ran.


In [ ]:
rows = []
for ev in all_evals:
    row = {"Model": ev['name']}
    for k, v in ev['test_metrics'].items():
        if k != 'Threshold':
            row[k] = round(float(v), 4)
    row['AUROC_CI'] = f"[{ev['cis']['AUROC_CI'][0]:.3f}, {ev['cis']['AUROC_CI'][1]:.3f}]"
    row['AUPRC_CI'] = f"[{ev['cis']['AUPRC_CI'][0]:.3f}, {ev['cis']['AUPRC_CI'][1]:.3f}]"
    row['Calibration'] = ev.get('calibration', 'raw')
    row['Time(s)'] = ev.get('time', np.nan)
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('Model')
print("═" * 100)
print("FINAL COMPARISON - TEST SET")
print("═" * 100)
print(comparison_df.to_string())
comparison_df.to_csv(f"{RESULTS_DIR}/model_comparison.csv")

val_rows = []
for ev in all_evals:
    row = {"Model": ev['name']}
    for k, v in ev['val_metrics'].items():
        if k != 'Threshold':
            row[k] = round(float(v), 4)
    row['Calibration'] = ev.get('calibration', 'raw')
    val_rows.append(row)

val_df_out = pd.DataFrame(val_rows).set_index('Model')
print("\nVALIDATION METRICS")
print(val_df_out.to_string())
val_df_out.to_csv(f"{RESULTS_DIR}/val_metrics.csv")

## 15. Save Predictions

One CSV per model plus a merged folder for later analysis.


In [ ]:
for name, bundle in pred_store.items():
    safe_name = name.lower().replace(" ", "_").replace("+", "plus").replace(".", "").replace("-", "_")
    bundle['val'].to_csv(f"{RESULTS_DIR}/{safe_name}_val_predictions.csv", index=False)
    bundle['test'].to_csv(f"{RESULTS_DIR}/{safe_name}_test_predictions.csv", index=False)

if 'best_xgb' in globals() and best_xgb is not None:
    xgb_imp = pd.DataFrame({"feature": feat_cols, "importance": best_xgb.feature_importances_})
    xgb_imp = xgb_imp.sort_values("importance", ascending=False).reset_index(drop=True)
    xgb_imp.to_csv(f"{RESULTS_DIR}/xgb_feature_importance.csv", index=False)

if 'best_lgbm' in globals() and HAS_LGBM and best_lgbm is not None:
    lgb_imp = pd.DataFrame({"feature": feat_cols, "importance": best_lgbm.feature_importances_})
    lgb_imp = lgb_imp.sort_values("importance", ascending=False).reset_index(drop=True)
    lgb_imp.to_csv(f"{RESULTS_DIR}/lgbm_feature_importance.csv", index=False)

if 'best_cat' in globals() and HAS_CAT and best_cat is not None:
    cat_imp = pd.DataFrame({"feature": feat_cols, "importance": best_cat.get_feature_importance()})
    cat_imp = cat_imp.sort_values("importance", ascending=False).reset_index(drop=True)
    cat_imp.to_csv(f"{RESULTS_DIR}/catboost_feature_importance.csv", index=False)

print(f"Saved outputs to: {RESULTS_DIR}")

## 16. Visualization - Publication-Ready Plots


### Plot 1: ROC Curves

In [ ]:
plot_specs = []
color_map = {
    "LR": "#2196F3",
    "XGB": "#4CAF50",
    "LGBM": "#2E7D32",
    "LSTM": "#FF9800",
    "BiLSTM": "#8E24AA",
    "TF": "#E91E63",
    "TabPFN-2.5": "#795548",
    "Ensemble": "#000000",
}

# Map full model names (as found in pred_store) to their abbreviated display names (for color_map lookup)
model_name_to_display_name = {
    "Logistic Regression": "LR",
    "XGBoost": "XGB",
    "LightGBM": "LGBM",
    "CatBoost": "CatBoost",
    "LSTM": "LSTM",
    "BiLSTM + Attention": "BiLSTM",
    "Transformer Encoder": "TF",
    "Weighted Rank Ensemble": "Ensemble",
}

# Dynamically populate plot_specs
for model_full_name, bundle in pred_store.items():
    display_name = model_name_to_display_name.get(model_full_name)
    if display_name is None:
        # Skip if model name is not in our display map (e.g., if TabPFN was skipped)
        continue

    y_true_for_model = None
    # Assign the correct y_true based on whether it's a flat model or sequence model
    if model_full_name in ["Logistic Regression", "XGBoost", "LightGBM", "CatBoost"]:
        y_true_for_model = y_te
    elif model_full_name in ["LSTM", "BiLSTM + Attention", "Transformer Encoder"]:
        y_true_for_model = yte_seq
    elif model_full_name == "Weighted Rank Ensemble":
        # Ensemble y_true is part of its predictions df
        y_true_for_model = bundle["test"]["y_true"].values
    # Add TabPFN if it successfully ran and is in pred_store
    elif model_full_name == "TabPFN-2.5":
        if "TabPFN-2.5" in pred_store.keys(): # Check if it actually ran
            y_true_for_model = y_te

    if y_true_for_model is not None:
        y_prob = bundle["test"]["prob"].values
        plot_specs.append((display_name, y_true_for_model, y_prob))

fig, ax = plt.subplots(figsize=(8, 6.5))
for key, y_true, y_prob in plot_specs:
    if len(np.unique(y_true)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color_map.get(key), label=f"{key} ({auc:.3f})")

ax.plot([0, 1], [0, 1], '--', lw=1, color='gray')
ax.set_title("ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/roc_curves.png", bbox_inches='tight')
plt.show()

### Plot 2: Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
for key, y_true, y_prob in plot_specs:
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax.plot(rec, prec, lw=2, color=color_map.get(key, None), label=f"{key} ({ap:.3f})")

baseline_rate = y_te.mean()
ax.axhline(baseline_rate, linestyle='--', linewidth=1, color='gray', label=f"No-skill ({baseline_rate:.3f})")
ax.set_title("Precision-Recall Curves")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pr_curves.png", bbox_inches='tight')
plt.show()

### Plot 3: Feature Importance

In [ ]:
fi_df = None
fi_name = None

# It's good to know the length of feat_cols
print(f"DEBUG: Length of feat_cols: {len(feat_cols)}")

# Try LightGBM first
if 'best_lgbm' in globals() and HAS_LGBM and best_lgbm is not None:
    lgbm_importances = best_lgbm.feature_importances_
    print(f"DEBUG: Length of best_lgbm.feature_importances_: {len(lgbm_importances)}")
    if len(feat_cols) == len(lgbm_importances):
        fi_df = pd.DataFrame({"feature": feat_cols, "importance": lgbm_importances})
        fi_name = "LightGBM"
    else:
        print(f"WARNING: LightGBM feature importances length ({len(lgbm_importances)}) does not match feat_cols length ({len(feat_cols)}). Skipping LightGBM feature importance plot.")

# If LightGBM failed, try CatBoost
if fi_df is None and 'best_cat' in globals() and HAS_CAT and best_cat is not None:
    cat_importances = best_cat.get_feature_importance()
    print(f"DEBUG: Length of best_cat.get_feature_importance(): {len(cat_importances)}")
    if len(feat_cols) == len(cat_importances):
        fi_df = pd.DataFrame({"feature": feat_cols, "importance": cat_importances})
        fi_name = "CatBoost"
    else:
        print(f"WARNING: CatBoost feature importances length ({len(cat_importances)}) does not match feat_cols length ({len(feat_cols)}). Skipping CatBoost feature importance plot.")

# If CatBoost also failed, try XGBoost
if fi_df is None and 'best_xgb' in globals() and best_xgb is not None:
    xgb_importances = best_xgb.feature_importances_
    print(f"DEBUG: Length of best_xgb.feature_importances_: {len(xgb_importances)}")
    if len(feat_cols) == len(xgb_importances):
        fi_df = pd.DataFrame({"feature": feat_cols, "importance": xgb_importances})
        fi_name = "XGBoost"
    else:
        print(f"WARNING: XGBoost feature importances length ({len(xgb_importances)}) does not match feat_cols length ({len(feat_cols)}). Skipping XGBoost feature importance plot.")

if fi_df is not None:
    fi_df = fi_df.sort_values("importance", ascending=False).head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(fi_df['feature'], fi_df['importance'])
    ax.set_title(f"Top 20 Feature Importance - {fi_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/feature_importance_top20.png", bbox_inches='tight')
    plt.show()
else:
    print("No tree-based feature importance available with matching feature list length.")

In [ ]:
import seaborn as sns

# Plot 1: Distribution of Prediction Probabilities
plt.figure(figsize=(10, 6))
for model_name in ["Logistic Regression", "XGBoost", "BiLSTM + Attention", "Weighted Rank Ensemble"]:
    if model_name in pred_store:
        sns.kdeplot(pred_store[model_name]["test"]["prob"], label=model_name, fill=True, alpha=0.2)

plt.title("Distribution of Predicted Probabilities (Test Set)")
plt.xlabel("Probability Score")
plt.ylabel("Density")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Plot 2: Confusion Matrix for the Weighted Rank Ensemble
ens_bundle = pred_store["Weighted Rank Ensemble"]["test"]
y_true_ens = ens_bundle["y_true"]
y_prob_ens = ens_bundle["prob"]
th_ens = ens_eval['val_metrics']['Threshold']
y_pred_ens = (y_prob_ens >= th_ens).astype(int)

cm = confusion_matrix(y_true_ens, y_pred_ens)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Distress", "Distress"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title("Confusion Matrix: Weighted Rank Ensemble (Test Set)")
plt.show()

In [ ]:
# Plot 3: Risk Timeline for a specific country (e.g., Argentina)
sample_iso = "ARG"
plt.figure(figsize=(12, 6))

for model_name in ["Logistic Regression", "BiLSTM + Attention", "Weighted Rank Ensemble"]:
    if model_name in pred_store:
        df_sub = pred_store[model_name]["test"]
        df_sub = df_sub[df_sub['iso3'] == sample_iso].sort_values('year_month')
        plt.plot(df_sub['year_month'], df_sub['prob'], label=f"{model_name} Prob")

# Add actual labels
y_true_sub = pred_store["Weighted Rank Ensemble"]["test"]
y_true_sub = y_true_sub[y_true_sub['iso3'] == sample_iso].sort_values('year_month')
plt.fill_between(y_true_sub['year_month'], 0, y_true_sub['y_true'], color='red', alpha=0.2, label="Actual Distress Window")

plt.title(f"Predicted Distress Probability Timeline: {sample_iso}")
plt.xlabel("Date")
plt.ylabel("Probability / Event")
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.show()

### Plot 4: Model Comparison

In [ ]:
metrics_to_plot = ['AUROC', 'AUPRC', 'F1', 'Recall']
plot_df = comparison_df.reset_index()

fig, axes = plt.subplots(1, 4, figsize=(17, 5))
for ax, metric in zip(axes, metrics_to_plot):
    vals = plot_df[metric].values
    ax.bar(plot_df['Model'], vals)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=60)
    ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/model_comparison_bars.png", bbox_inches='tight')
plt.show()

### Plot 5: Lead-Time Analysis

In [ ]:
lead_rows = []
for ev in all_evals:
    lead_rows.append({
        "Model": ev['name'],
        "Median Lead (months)": np.median(ev['lead_times']) if len(ev['lead_times']) > 0 else 0.0,
        "Detected Events": int(ev['lead_df']['detected'].sum()) if len(ev['lead_df']) > 0 else 0
    })

lead_df_plot = pd.DataFrame(lead_rows)
print(lead_df_plot.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(lead_df_plot['Model'], lead_df_plot['Median Lead (months)'])
ax.set_title("Median Lead Time by Model")
ax.set_ylabel("Months")
ax.tick_params(axis='x', rotation=60)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/lead_time_bar.png", bbox_inches='tight')
plt.show()

## 17. Final Summary

In [ ]:
print("═" * 90)
print("COMPLETE - FINAL IMPROVED BASELINES")
print("═" * 90)
print(f"Models evaluated: {list(pred_store.keys())}")
print(f"Flat features used: {len(feat_cols)}")
print(f"Augmented sequence shape: {SEQ_LEN} x {N_FEAT}")
print(f"Results directory: {RESULTS_DIR}")

best_test_auprc = comparison_df['AUPRC'].astype(float).idxmax()
best_test_auroc = comparison_df['AUROC'].astype(float).idxmax()
best_test_f1    = comparison_df['F1'].astype(float).idxmax()
print(f"Best test AUPRC model: {best_test_auprc}")
print(f"Best test AUROC model: {best_test_auroc}")
print(f"Best test F1 model: {best_test_f1}")